# v16.1 — detektor AF 2 kelas dari tahap 1 v16 (ECGFounder tanpa penyetelan)

Rancangan dikunci sebelum metrik baru dihitung: [RANCANGAN.md](RANCANGAN.md) (sha256 dan waktu di `../logs/v16.1_praregistrasi.log`).
Logika di `deteksi_af.py`. Keluaran hanya **Normal vs AF** (AFIB atau AFL digabung). Tidak ada training dan tidak ada parameter yang disetel.

| Sistem | Keputusan AF |
|---|---|
| **D-0 (primer)** | ECGFounder zero-shot: log((pAF + pAFL) / pN) > 0 |
| **D-L** | D-0, atau v3.1 menyebut AFL ("perlu dicek" v16 dihitung sebagai alarm) |
| v3.1, BiGRU bawaan (v11.1), v1 | bias 3 kelas cross-fit (v1: argmax), lalu AFIB + AFL digabung (aturan v11.1) |

In [1]:
import json, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import deteksi_af as A
from deteksi_af import D, DK, E13, Ev, H16, K33, T, cv


def ci3(v):
    return f"{v[0]:.3f} [{v[1]:.2f}–{v[2]:.2f}]"


def dl(v):
    return f"Δ {v[0]:+.3f} [{v[1]:+.3f}…{v[2]:+.3f}], P = {v[3]:.3f} → {Ev.vonis(v[3])}"


d = D.muat()
R = len(d.folds)
y2 = (d.y > 0).astype(np.int64)
d1 = A.d1_afdb()
p3, oof = A.pembanding(d)
pred2 = {k: A.ke2(v) for k, v in p3.items()}
pred2.update(A.sistem(d1, p3["v3.1"]))
print("window AFDB:", len(d.y), "| AF:", int(y2.sum()), "| ulangan:", R)

window AFDB: 165978 | AF: 66978 | ulangan: 5


## 1. AFDB 2 kelas

v3.1 dan BiGRU di-assert sama dengan ringkasan v11.1; D-0 di-assert sama dengan H2b v16.

In [2]:
W = cv.bootstrap_weights(d.strata, Ev.N_BOOT, seed=D.SEED)
hasil, boot = {}, {}
for k, p in pred2.items():
    hasil[k], boot[k] = DK.ringkas(DK.cm_pasien(y2, p, d.records, d.patients), W)
r11 = json.loads((D.REPO / "cache/v11.1/dua_kelas.json").read_text())["aturan"]["gabung"]["sistem"]
r16 = json.loads((H16.CACHE / "ringkasan.json").read_text())
assert np.allclose(hasil["v3.1"]["Macro-F1"], r11["v3.1"]["Macro-F1"])
assert np.allclose(hasil["BiGRU bawaan (v11.1)"]["Macro-F1"], r11["BiGRU bawaan"]["Macro-F1"])
assert np.allclose(hasil["D-0"]["Macro-F1"], r16["dua_kelas"]["H-0"]["Macro-F1"])
banding = {k: DK.banding(hasil[k], hasil["v3.1"], boot[k], boot["v3.1"]) for k in hasil if k != "v3.1"}
banding["D-L vs D-0"] = DK.banding(hasil["D-L"], hasil["D-0"], boot["D-L"], boot["D-0"])
banding["D-0 vs BiGRU"] = DK.banding(hasil["D-0"], hasil["BiGRU bawaan (v11.1)"], boot["D-0"], boot["BiGRU bawaan (v11.1)"])
display(pd.DataFrame({k: {m: ci3(v) for m, v in r.items()} for k, r in hasil.items()}).T.rename_axis("AFDB 2 kelas"))
pd.DataFrame({k: {m: dl(v) for m, v in b.items()} for k, b in banding.items()}).T.rename_axis("vs v3.1 (kecuali 2 baris terakhir)")

,Macro-F1,F1 AF1,Se,Sp,PPV,F1 Normal
AFDB 2 kelas,,,,,,
v1 (argmax),0.922 [0.88–0.95],0.910 [0.87–0.94],0.954 [0.94–0.97],0.903 [0.82–0.95],0.870 [0.79–0.93],0.934 [0.88–0.96]
v3.1,0.926 [0.88–0.95],0.913 [0.87–0.94],0.934 [0.91–0.96],0.924 [0.84–0.97],0.892 [0.81–0.95],0.938 [0.89–0.97]
BiGRU bawaan (v11.1),0.938 [0.91–0.96],0.926 [0.90–0.95],0.926 [0.90–0.95],0.950 [0.90–0.98],0.926 [0.87–0.97],0.950 [0.92–0.97]
D-0,0.951 [0.92–0.98],0.940 [0.91–0.97],0.893 [0.83–0.95],0.995 [0.99–1.00],0.992 [0.99–1.00],0.963 [0.94–0.98]
D-L,0.956 [0.93–0.98],0.947 [0.91–0.98],0.905 [0.85–0.96],0.995 [0.99–1.00],0.992 [0.99–1.00],0.966 [0.94–0.99]


,Macro-F1,F1 AF1,Se,Sp
vs v3.1 (kecuali 2 baris terakhir),,,,
v1 (argmax),"Δ -0.004 [-0.013…+0.005], P = 0.195 → tidak te...","Δ -0.003 [-0.014…+0.007], P = 0.298 → tidak te...","Δ +0.020 [+0.009…+0.030], P = 1.000 → lebih baik","Δ -0.020 [-0.033…-0.010], P = 0.000 → lebih buruk"
BiGRU bawaan (v11.1),"Δ +0.012 [+0.002…+0.030], P = 0.996 → lebih baik","Δ +0.013 [+0.001…+0.032], P = 0.990 → lebih baik","Δ -0.008 [-0.016…+0.000], P = 0.028 → lebih buruk","Δ +0.026 [+0.010…+0.058], P = 1.000 → lebih baik"
D-0,"Δ +0.026 [-0.008…+0.070], P = 0.920 → tidak te...","Δ +0.028 [-0.013…+0.077], P = 0.892 → tidak te...","Δ -0.040 [-0.086…-0.002], P = 0.017 → lebih buruk","Δ +0.071 [+0.029…+0.150], P = 1.000 → lebih baik"
D-L,"Δ +0.031 [-0.003…+0.076], P = 0.953 → lebih baik","Δ +0.034 [-0.006…+0.083], P = 0.939 → tidak te...","Δ -0.028 [-0.074…+0.013], P = 0.090 → tidak te...","Δ +0.071 [+0.029…+0.150], P = 1.000 → lebih baik"
D-L vs D-0,"Δ +0.005 [-0.000…+0.011], P = 0.966 → lebih baik","Δ +0.006 [-0.000…+0.014], P = 0.970 → lebih baik","Δ +0.012 [+0.001…+0.026], P = 1.000 → lebih baik","Δ -0.001 [-0.002…-0.000], P = 0.000 → lebih buruk"
D-0 vs BiGRU,"Δ +0.014 [-0.012…+0.043], P = 0.833 → tidak te...","Δ +0.014 [-0.018…+0.048], P = 0.804 → tidak te...","Δ -0.033 [-0.079…+0.006], P = 0.048 → lebih buruk","Δ +0.045 [+0.017…+0.093], P = 1.000 → lebih baik"


## 2. Tanpa 04936 (H2), per pasien (H3), alarm Normal per pita detak, episode AF

In [3]:
tanpa, boot_m = A.tanpa_04936(d, d1, oof)
tanpa_banding = {k: DK.banding(tanpa[k], tanpa["v3.1"], boot_m[k], boot_m["v3.1"]) for k in tanpa if k != "v3.1"}
display(pd.DataFrame({k: {m: ci3(v) for m, v in r.items() if m in ("Macro-F1", "Se", "Sp", "PPV")} for k, r in tanpa.items()}).T.rename_axis("tanpa 04936"))
display(pd.DataFrame({k: {m: dl(v) for m, v in b.items()} for k, b in tanpa_banding.items()}).T.rename_axis("tanpa 04936, vs v3.1"))
pp = {k: A.per_pasien(d, pred2[k]) for k in ("v3.1", "BiGRU bawaan (v11.1)", "D-0", "D-L")}
turun = {p: round(pp["v3.1"][p]["se"] - pp["D-0"][p]["se"], 3) for p in pp["D-0"]
         if pp["D-0"][p]["se"] is not None and pp["v3.1"][p]["se"] - pp["D-0"][p]["se"] > 0.20}
display(pd.DataFrame({p: {**{f"Se {k}": v[p]["se"] for k, v in pp.items()}, **{f"alarm {k}": v[p]["alarm"] for k, v in pp.items()},
                          "n AF": pp["D-0"][p]["n_af"], "n Normal": pp["D-0"][p]["n_normal"]} for p in pp["D-0"]}).T.round(3).rename_axis("per pasien"))
print("pasien dengan sensitivitas D-0 turun > 0,20 dari v3.1:", turun)
alarm_n = {k: A.alarm_normal(d, p) for k, p in pred2.items()}
display(pd.DataFrame({k: {b: f"{x['alarm']*100:.1f}% (n={x['n']})" for b, x in v.items()} for k, v in alarm_n.items()}).T.rename_axis("alarm pada Normal AFDB"))
ep = {k: A.episode_af(d, pred2[k]) for k in ("v3.1", "BiGRU bawaan (v11.1)", "D-0", "D-L")}
pd.DataFrame(ep).T.round(3).rename_axis("episode AF (EC57)")

,Macro-F1,Se,Sp,PPV
tanpa 04936,,,,
v1 (argmax),0.922 [0.88–0.95],0.959 [0.94–0.98],0.902 [0.83–0.95],0.860 [0.78–0.92]
v3.1,0.927 [0.88–0.95],0.933 [0.90–0.96],0.928 [0.86–0.97],0.891 [0.80–0.95]
BiGRU bawaan (v11.1),0.938 [0.91–0.96],0.918 [0.89–0.95],0.956 [0.92–0.98],0.928 [0.88–0.97]
D-0,0.954 [0.92–0.98],0.895 [0.83–0.96],0.995 [0.99–1.00],0.992 [0.99–1.00]
D-L,0.956 [0.93–0.98],0.901 [0.84–0.96],0.995 [0.99–1.00],0.991 [0.98–1.00]


,Macro-F1,F1 AF1,Se,Sp
"tanpa 04936, vs v3.1",,,,
v1 (argmax),"Δ -0.005 [-0.016…+0.005], P = 0.154 → tidak te...","Δ -0.005 [-0.018…+0.008], P = 0.253 → tidak te...","Δ +0.026 [+0.011…+0.041], P = 1.000 → lebih baik","Δ -0.026 [-0.040…-0.014], P = 0.000 → lebih buruk"
BiGRU bawaan (v11.1),"Δ +0.011 [-0.003…+0.031], P = 0.915 → tidak te...","Δ +0.012 [-0.004…+0.034], P = 0.887 → tidak te...","Δ -0.015 [-0.030…-0.001], P = 0.020 → lebih buruk","Δ +0.027 [+0.010…+0.060], P = 1.000 → lebih baik"
D-0,"Δ +0.027 [-0.008…+0.070], P = 0.930 → tidak te...","Δ +0.030 [-0.012…+0.080], P = 0.910 → tidak te...","Δ -0.038 [-0.087…+0.002], P = 0.029 → lebih buruk","Δ +0.067 [+0.028…+0.138], P = 1.000 → lebih baik"
D-L,"Δ +0.029 [-0.005…+0.072], P = 0.953 → lebih baik","Δ +0.033 [-0.008…+0.083], P = 0.936 → tidak te...","Δ -0.032 [-0.077…+0.004], P = 0.049 → lebih buruk","Δ +0.067 [+0.027…+0.137], P = 1.000 → lebih baik"


,Se v3.1,Se BiGRU bawaan (v11.1),Se D-0,Se D-L,alarm v3.1,alarm BiGRU bawaan (v11.1),alarm D-0,alarm D-L,n AF,n Normal
per pasien,,,,,,,,,,
04015,1.000,0.969,0.938,0.938,0.094,0.073,0.007,0.007,32.0,7219.0
04043,0.984,0.967,0.921,0.935,0.045,0.040,0.012,0.021,1441.0,5624.0
04048,0.984,0.968,0.800,0.800,0.033,0.023,0.009,0.009,25.0,7074.0
04126,0.953,0.916,1.000,1.000,0.065,0.051,0.002,0.002,119.0,7043.0
04746,0.960,0.958,0.994,0.994,0.007,0.003,0.001,0.001,3888.0,3438.0
04908,0.993,0.982,0.914,0.981,0.006,0.004,0.000,0.000,650.0,6689.0
04936,0.847,0.856,0.870,0.946,0.017,0.011,0.010,0.010,5932.0,1323.0
05091,0.933,0.933,0.667,0.667,0.023,0.006,0.000,0.000,9.0,7210.0
05121,0.903,0.907,0.687,0.687,0.041,0.036,0.017,0.017,4516.0,2612.0


pasien dengan sensitivitas D-0 turun > 0,20 dari v3.1: {'05091': 0.267, '05121': 0.216, '08378': 0.27}


,< 100,100–110,110–120,≥ 120,teratur 80–95
alarm pada Normal AFDB,,,,,
v1 (argmax),8.8% (n=89566),14.2% (n=8766),66.2% (n=568),94.4% (n=100),0.3% (n=4968)
v3.1,6.7% (n=89566),12.7% (n=8766),59.0% (n=568),95.2% (n=100),0.1% (n=4968)
BiGRU bawaan (v11.1),4.1% (n=89566),10.0% (n=8766),52.6% (n=568),86.6% (n=100),0.1% (n=4968)
D-0,0.3% (n=89566),0.9% (n=8766),12.0% (n=568),33.0% (n=100),0.0% (n=4968)
D-L,0.3% (n=89566),1.0% (n=8766),19.6% (n=568),37.8% (n=100),0.0% (n=4968)


,se_episode,ppv_episode,se_durasi,ppv_durasi,n_ref,n_det,palsu_per_jam
episode AF (EC57),,,,,,,
v3.1,0.970,0.567,0.934,0.892,895.0,6099.4,11.442
BiGRU bawaan (v11.1),0.949,0.631,0.926,0.926,895.0,5857.6,9.304
D-0,0.914,0.946,0.893,0.992,895.0,4341.0,1.019
D-L,0.922,0.941,0.905,0.992,895.0,4372.6,1.116


## 3. Sinus takikardia asli (butir 9) dan D-L

Urutan window: alarm D-0 dan v3.1 di-assert sama dengan v16. BiGRU bawaan = 25 model fold v3.3 dengan bias fold-nya.

In [4]:
L = H16.latihan()
J = L["J"]
_, d1l = H16.tahap1(L["logit"])
amax = L["p31"].argmax(1)
kl = {"v3.1 paket": amax != 0, "D-0": d1l > 0, "D-L": (d1l > 0) | (amax == 2)}
penuh = {k: H16.kelas_latihan(L, v.astype(np.int64)) for k, v in kl.items()}
penuh["BiGRU bawaan (v11.1)"] = np.load(K33.CACHE / "prediksi.npz")["v11_bigru"]
hr_m, subj_m, src_m = (J[k][J["ok"]][L["mask"]] for k in ("hr", "subjek", "sumber"))
m120 = hr_m >= 120
lat = {}
for k, v in penuh.items():
    a = (v[:, L["mask"]] != 0).mean(0)
    t, ci, _ = K33._boot_subjek(subj_m[m120], a[m120])
    lat[k] = dict(alarm=t, ci=ci.tolist(), n=int(m120.sum()), per_sumber={s: float(a[m120 & (src_m == s)].mean()) for s in np.unique(src_m[m120])},
                  pita=K33.alarm_per_pita(J, v, L["mask"]))
for k in ("D-0", "D-L", "BiGRU bawaan (v11.1)"):
    lat[k]["banding"] = K33.banding_alarm(J, penuh[k], penuh["v3.1 paket"], L["mask"])
assert np.isclose(lat["D-0"]["alarm"], r16["h1_latihan"]["H-0"]["alarm"]) and np.isclose(lat["v3.1 paket"]["alarm"], r16["h1_latihan"]["v3.1 paket"]["alarm"])
display(pd.DataFrame({k: {"alarm ≥120 bpm": f"{v['alarm']*100:.1f}% [{v['ci'][0]*100:.1f}–{v['ci'][1]*100:.1f}]",
                          **{s: f"{x*100:.0f}%" for s, x in v["per_sumber"].items()}} for k, v in lat.items()}).T)
pd.DataFrame({k: {p: f"{x['alarm']*100:.0f}%" for p, x in v["pita"].items()} for k, v in lat.items()}).T.rename_axis("alarm per pita (latihan fisik)")

,alarm ≥120 bpm,EPFL,EPHNOGRAM,Wrist
v3.1 paket,98.4% [96.5–99.6],100%,98%,100%
D-0,0.1% [0.0–0.3],0%,0%,0%
D-L,74.6% [61.2–84.9],90%,74%,50%
BiGRU bawaan (v11.1),91.0% [87.4–93.9],100%,91%,84%


,0–100 bpm,100–110 bpm,110–120 bpm,120–130 bpm,130–140 bpm,140–150 bpm,150–∞ bpm
alarm per pita (latihan fisik),,,,,,,
v3.1 paket,22%,16%,62%,97%,99%,99%,100%
D-0,1%,1%,0%,0%,0%,0%,0%
D-L,1%,4%,40%,60%,68%,84%,97%
BiGRU bawaan (v11.1),22%,12%,29%,80%,96%,99%,99%


## 4. Data eksternal 2 kelas (H1)

Pembanding: paket v3.1 (argmax, AFIB + AFL digabung). Bootstrap pasien berpasangan 2.000×.

In [5]:
P = H16.ptbxl()
_, d1p = H16.tahap1(P["logit"])
y2p, ap = (P["y"] > 0).astype(np.int64), P["p31"].argmax(1)
kp = {"v3.1 paket": ap != 0, "D-0": d1p > 0, "D-L": (d1p > 0) | (ap == 2)}
ptb = {k: A.banding2_pasien(y2p, kp[k], kp["v3.1 paket"], P["pasien"]) for k in ("D-0", "D-L")}
ptb_kelas = {k: {"AFIB": float(v[P["y"] == 1].mean()), "AFL": float(v[P["y"] == 2].mean()), "STACH": float(v[P["st"]].mean()),
                 "sinus lain": float(v[(P["y"] == 0) & ~P["st"]].mean())} for k, v in kp.items()}
I = H16.icentia()
_, d1i = H16.tahap1(I["logit"])
y2i, ai = (I["y"] > 0).astype(np.int64), I["p31"].argmax(1)
ki = {"v3.1 paket": ai != 0, "D-0": d1i > 0, "D-L": (d1i > 0) | (ai == 2)}
ico = {k: A.banding2_pasien(y2i, ki[k], ki["v3.1 paket"], I["pasien"]) for k in ("D-0", "D-L")}
ico_kelas = {k: {"AFIB": float(v[I["y"] == 1].mean()), "AFL": float(v[I["y"] == 2].mean()), "Normal": float(v[I["y"] == 0].mean())} for k, v in ki.items()}
for nama, hsl in (("PTB-XL", ptb), ("Icentia11k", ico)):
    display(pd.DataFrame({k: {**{f"{m} (sistem / v3.1)": f"{v['titik_a'][m][0]:.3f} / {v['titik_b'][m][0]:.3f}" for m in ("Macro-F1", "Se", "Sp")},
                              **{f"Δ {m}": dl(v[m]) for m in ("Macro-F1", "Se", "Sp")}} for k, v in hsl.items()}).T.rename_axis(nama))
display(pd.DataFrame(ptb_kelas).T.round(3).rename_axis("PTB-XL: proporsi alarm AF per kelas asli"))
pd.DataFrame(ico_kelas).T.round(3).rename_axis("Icentia11k: proporsi alarm AF per kelas asli")

,Macro-F1 (sistem / v3.1),Se (sistem / v3.1),Sp (sistem / v3.1),Δ Macro-F1,Δ Se,Δ Sp
PTB-XL,,,,,,
D-0,0.944 / 0.817,0.863 / 0.917,0.995 / 0.935,"Δ +0.127 [+0.116…+0.137], P = 1.000 → lebih baik","Δ -0.054 [-0.073…-0.034], P = 0.000 → lebih buruk","Δ +0.060 [+0.056…+0.063], P = 1.000 → lebih baik"
D-L,0.926 / 0.817,0.879 / 0.917,0.988 / 0.935,"Δ +0.109 [+0.100…+0.119], P = 1.000 → lebih baik","Δ -0.038 [-0.055…-0.020], P = 0.000 → lebih buruk","Δ +0.052 [+0.049…+0.056], P = 1.000 → lebih baik"


,Macro-F1 (sistem / v3.1),Se (sistem / v3.1),Sp (sistem / v3.1),Δ Macro-F1,Δ Se,Δ Sp
Icentia11k,,,,,,
D-0,0.823 / 0.858,0.767 / 0.871,0.987 / 0.882,"Δ -0.035 [-0.047…-0.023], P = 0.000 → lebih buruk","Δ -0.105 [-0.120…-0.088], P = 0.000 → lebih buruk","Δ +0.105 [+0.096…+0.113], P = 1.000 → lebih baik"
D-L,0.839 / 0.858,0.798 / 0.871,0.973 / 0.882,"Δ -0.019 [-0.028…-0.009], P = 0.000 → lebih buruk","Δ -0.074 [-0.087…-0.061], P = 0.000 → lebih buruk","Δ +0.091 [+0.084…+0.099], P = 1.000 → lebih baik"


,AFIB,AFL,STACH,sinus lain
PTB-XL: proporsi alarm AF per kelas asli,,,,
v3.1 paket,0.917,0.92,0.297,0.055
D-0,0.880,0.36,0.016,0.005
D-L,0.885,0.72,0.182,0.005


,AFIB,AFL,Normal
Icentia11k: proporsi alarm AF per kelas asli,,,
v3.1 paket,0.940,0.836,0.118
D-0,0.853,0.722,0.013
D-L,0.854,0.768,0.027


## 5. Kurva operasi ambang (H5, deskriptif; tidak ada ambang yang dipilih)

In [6]:
krv = A.kurva(d, d1, d1l[L["hr"] >= 120], d1p, P["y"], P["st"])
pd.DataFrame(krv).set_index("t").round(3)

,se_afdb,sp_afdb,mf1_afdb,alarm_st,alarm_stach,se_ptb,sp_ptb
t,,,,,,,
-4,0.989,0.859,0.911,0.259,0.067,0.979,0.801
-3,0.979,0.909,0.936,0.075,0.051,0.969,0.846
-2,0.962,0.971,0.966,0.020,0.037,0.943,0.938
-1,0.935,0.992,0.968,0.005,0.025,0.912,0.988
0,0.893,0.995,0.951,0.001,0.016,0.863,0.995
1,0.818,0.997,0.919,0.000,0.009,0.746,0.998
2,0.604,0.999,0.817,0.000,0.004,0.514,0.999
3,0.232,1.000,0.585,0.000,0.000,0.207,1.000
4,0.047,1.000,0.423,0.000,0.000,0.040,1.000


### 5b. Tambahan setelah hasil: kurva operasi di Icentia11k

Dicatat di RANCANGAN bagian 8. Deskriptif; D-0 tetap t = 0 dan kriteria tidak berubah.

In [7]:
krv_ico = []
for t in A.AMBANG_KURVA:
    a = d1i > t
    cm = np.bincount(y2i * 2 + a, minlength=4).reshape(2, 2).astype(float)
    krv_ico.append(dict(t=int(t), se=float(a[y2i == 1].mean()), sp=float((~a[y2i == 0]).mean()), mf1=float(DK.f1_makro_cm(cm)),
                        afl=float(a[I["y"] == 2].mean()), afib=float(a[I["y"] == 1].mean())))
mv = DK.metrik(np.bincount(y2i * 2 + ki["v3.1 paket"], minlength=4).reshape(2, 2).astype(float))
print("paket v3.1 di Icentia11k (pembanding):", {m: round(float(mv[m]), 3) for m in ("Macro-F1", "Se", "Sp")})
pd.DataFrame(krv_ico).set_index("t").round(3)

paket v3.1 di Icentia11k (pembanding): {'Macro-F1': 0.858, 'Se': 0.871, 'Sp': 0.882}


,se,sp,mf1,afl,afib
t,,,,,
-4,0.962,0.726,0.863,0.946,0.992
-3,0.932,0.786,0.864,0.905,0.983
-2,0.894,0.908,0.883,0.856,0.965
-1,0.843,0.972,0.870,0.800,0.924
0,0.767,0.987,0.823,0.722,0.853
1,0.617,0.993,0.725,0.571,0.706
2,0.368,0.998,0.556,0.320,0.462
3,0.122,1.000,0.356,0.086,0.192
4,0.024,1.000,0.258,0.012,0.048


## 6. Fungsi prediksi: paritas dan latensi (H6)

In [8]:
import torch
pilih, w = A.jendela_paritas()
par = A.paritas(d1, pilih, w, "cpu")
lat_cpu = A.latensi(w, "cpu", batch=1, n=50)
lat_gpu = A.latensi(w, "cuda", batch=64, n=10) if torch.cuda.is_available() else None
print("paritas:", par)
print("latensi CPU:", lat_cpu)
print("latensi GPU:", lat_gpu)

paritas: {'n': 606, 'sama': 1.0, 'maks_selisih': 0.503476534054621, 'median_selisih': 0.03226792726214178, 'device': 'cpu'}
latensi CPU: {'device': 'cpu', 'batch': 1, 'ms_per_window': 29.038544998911675, 'p90': 29.73947710270295, 'threads': 6, 'gpu': None}
latensi GPU: {'device': 'cuda', 'batch': 64, 'ms_per_window': 0.7313253437928324, 'p90': 0.747059610944234, 'threads': 6, 'gpu': 'NVIDIA GeForce RTX 5060'}


In [9]:
kriteria = dict(afdb=bool(banding["D-0"]["Macro-F1"][3] > 0.05), tanpa_04936=bool(tanpa_banding["D-0"]["Macro-F1"][3] > 0.05),
                ptbxl=bool(ptb["D-0"]["Macro-F1"][3] > 0.05), icentia=bool(ico["D-0"]["Macro-F1"][3] > 0.05),
                sinus_takikardia=bool(lat["D-0"]["alarm"] <= 0.5 * lat["v3.1 paket"]["alarm"]))
kriteria["kandidat_uji_perangkat"] = all(kriteria.values())
ring = {"afdb": hasil, "banding": banding, "bersarang_v16": r16["dua_kelas"]["bersarang"], "tanpa_04936": tanpa,
        "tanpa_04936_banding": tanpa_banding, "per_pasien": pp, "se_turun_0_20": turun, "alarm_normal": alarm_n, "episode": ep,
        "latihan": lat, "ptbxl": ptb, "ptbxl_kelas": ptb_kelas, "icentia": ico, "icentia_kelas": ico_kelas, "kurva": krv, "kurva_icentia_tambahan": krv_ico,
        "paritas": par, "latensi": {"cpu": lat_cpu, "gpu": lat_gpu}, "kriteria": kriteria,
        "n": {"afdb": int(len(d.y)), "afdb_af": int(y2.sum()), "ptbxl": int(len(y2p)), "ptbxl_af": int(y2p.sum()), "icentia": int(len(y2i)),
              "icentia_af": int(y2i.sum()), "icentia_pasien": int(np.unique(I["pasien"]).size), "latihan_120": int(m120.sum())},
        "lingkungan": Ev._lingkungan()}
A.CACHE.mkdir(parents=True, exist_ok=True)
(A.CACHE / "ringkasan.json").write_text(json.dumps(ring, indent=1, default=float, ensure_ascii=False))
print("kriteria:", kriteria)
print("tersimpan:", A.CACHE / "ringkasan.json")

kriteria: {'afdb': True, 'tanpa_04936': True, 'ptbxl': True, 'icentia': False, 'sinus_takikardia': True, 'kandidat_uji_perangkat': False}
tersimpan: /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/cache/v16.1/ringkasan.json


## 7. Kesimpulan

Dijalankan penuh 6 November 2026 di PC (RTX 5060 untuk paritas/latensi; selebihnya dari cache, ±3 menit). Kriteria yang dikunci
(RANCANGAN bagian 5): AFDB, tanpa 04936, PTB-XL, dan sinus takikardia lolos; **Icentia11k gagal** → D-0 **belum kandidat** detektor AF
2 kelas untuk uji perangkat.

- **AFDB 2 kelas:** D-0 0,951 [0,92–0,98] vs v3.1 0,926 (P = 0,92, tidak terbedakan); tanpa 04936 0,954 vs 0,927 (P = 0,93).
  Sensitivitas 0,893 vs 0,934 (lebih buruk, P = 0,017), spesifisitas 0,995 vs 0,924. BiGRU bawaan v11.1 0,938 (lebih baik dari v3.1,
  P = 0,996); D-0 vs BiGRU P = 0,83.
- **Alarm palsu hampir hilang:** sinus takikardia asli ≥ 120 bpm 0,1% vs 98,4% (BiGRU 91,0%); Normal AFDB 100–110 bpm 0,9% vs 12,7%,
  110–120 bpm 12,0% vs 59,0%; episode AF palsu 11,4 → 1,0 per jam (+P episode 0,57 → 0,95, Se episode 0,97 → 0,91).
- **Eksternal:** PTB-XL 0,944 vs 0,817 (lebih baik; STACH 29,7% → 1,6%) tetapi AFL terdeteksi 36% vs 92%. Icentia11k 0,823 vs 0,858
  (lebih buruk): sensitivitas 0,767 vs 0,871, spesifisitas 0,987 vs 0,882.
- **Per pasien:** sensitivitas turun > 0,20 di 05091 (9 window AF), 05121, dan 08378 (AFL 2:1). Alarm pada Normal turun di semua pasien.
- **D-L** (D-0 + AFL v3.1): AFDB 0,956 (lebih baik dari v3.1, P = 0,953), tetapi alarm sinus takikardia asli naik lagi ke 74,6% dan
  Icentia tetap lebih buruk (0,839).
- **Kurva ambang (deskriptif):** t = −1 memberi sensitivitas AFDB 0,935 (≈ v3.1) dengan spesifisitas 0,992 dan alarm sinus takikardia
  0,5%; di Icentia11k (tambahan setelah hasil) Macro-F1 0,870. Terlihat setelah hasil di data yang sama, jadi harus dikunci dan diuji di
  data baru (CPSC 2021 belum pernah dinilai; rekaman perangkat).
- **Fungsi prediksi:** `DetektorAF` di CPU float32 sama keputusannya 100% dengan cache v13 (606 window 08378; selisih d1 maks 0,50).
  Latensi ±29 ms/window di CPU (6 thread), 0,73 ms/window di GPU (batch 64).